# 03 — KnottedGraph vs Topoly: focused paper scaling

This notebook runs only the **two benchmark families used in the paper**:

1. **Crossing scaling across heterogeneous connected trivalent graphs.** At every projected crossing count $c$, the benchmark uses the same panel of **21 different connected trivalent graphs** with different $(V,E)$. The graph-size distribution is therefore held fixed while $c$ changes.
2. **Trivalent input-size scaling.** Planar $K_4$ components are used to vary the total vertex count $V$ in a simple controlled family.

The old throughput, edge-theta, prism-$V$, prism-$E$, and random-cubic calculations are not run here.

For every individual sample, graph construction and graph-to-PD conversion happen **outside** the timed region. KnottedGraph and Topoly receive the exact same PD code. A successful long run is cached locally, so rerunning unchanged code/configuration loads the saved rows and proceeds directly to validation and plotting.


In [ ]:
from pathlib import Path
import csv, importlib.util, json, os, subprocess, sys

from tqdm.auto import tqdm

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
if not (ROOT / "src" / "knotted_graph").exists():
    raise RuntimeError("Run this notebook from inside the KnottedGraph checkout.")

SRC = ROOT / "src"
DEV = ROOT / "dev"
for path in (SRC, DEV):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

import knotted_graph

kg_path = Path(knotted_graph.__file__).resolve()
if SRC not in kg_path.parents:
    raise RuntimeError(f"A stale knotted_graph was imported from {kg_path}")

try:
    import topoly
except ImportError as exc:
    raise ImportError("Install Topoly first: pip install topoly") from exc

OUT = ROOT / "User_guide" / "benchmarks"
RES = OUT / "results_latest"
FIG = OUT / "figures_latest"
RES.mkdir(exist_ok=True)
FIG.mkdir(exist_ok=True)

print("KnottedGraph:", kg_path)
print("Topoly:", Path(topoly.__file__).resolve())


## 1. Configuration

The normal local paper run uses:

- **21 distinct connected trivalent graphs per crossing count** for the crossing figure;
- 10 independent deterministic embeddings per $V$ for the $K_4$ input-size figure;
- exactly one timed Yamada evaluation per framework per sample;
- a 120 s wall-time limit per framework/sample;
- censor-frontier stopping after repeated fully censored x-values.

GitHub Actions automatically uses a much smaller smoke profile. Smoke timings are only for correctness/execution validation and must not be used in the paper.


In [ ]:
IS_CI = os.environ.get("CI", "").lower() == "true"

PROFILE = "smoke" if IS_CI else "paper"
CROSSING_GRAPHS = 3 if IS_CI else 21
K4_EMBEDDINGS = 2 if IS_CI else 10
TIMEOUT_S = 10 if IS_CI else 120
BASE_SEED = 20260818
CENSOR_FRONTIER = 2

raw_csv = RES / "topoly_yamada_paper_scaling_raw.csv"
aggregate_csv = RES / "topoly_yamada_paper_scaling_aggregate.csv"

print(
    f"mode={PROFILE}, crossing graphs/c={CROSSING_GRAPHS}, "
    f"K4 embeddings/V={K4_EMBEDDINGS}, "
    f"timeout/framework/sample={TIMEOUT_S}s"
)


In [ ]:
def _load_module(path, name):
    spec = importlib.util.spec_from_file_location(name, path)
    if spec is None or spec.loader is None:
        raise RuntimeError(f"Could not load {path}")
    module = importlib.util.module_from_spec(spec)
    sys.modules[name] = module
    spec.loader.exec_module(module)
    return module


def run_streamed_benchmark(cmd, *, total, description):
    print("Running:", " ".join(map(str, cmd)))
    process = subprocess.Popen(
        cmd, cwd=ROOT, env=env, text=True,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=1,
    )
    if process.stdout is None:
        raise RuntimeError("Benchmark subprocess did not expose stdout.")

    streamed_rows = []
    summary_rows = None
    bar = tqdm(total=total, desc=description, unit="sample", dynamic_ncols=True)

    def _time_text(row, framework):
        status = row.get(f"{framework}_status", "?")
        value = row.get(f"{framework}_s")
        if status == "ok" and value is not None:
            return f"{float(value):.6f} s"
        if status == "timeout":
            return f"TIMEOUT (>={row.get('timeout_s', '?')} s)"
        if status == "error":
            return "ERROR"
        if status == "skipped_after_censor_frontier":
            return "SKIPPED"
        return status.upper()

    for raw_line in process.stdout:
        line = raw_line.rstrip()
        if not line:
            continue
        if line.startswith("SUMMARY="):
            summary_rows = json.loads(line[len("SUMMARY="):])
            continue
        if line.startswith("{"):
            try:
                row = json.loads(line)
            except json.JSONDecodeError:
                tqdm.write(line)
                continue
            if "family" in row:
                streamed_rows.append(row)
                sample_id = row.get("sample", row.get("embedding", "?"))
                meta = " ".join(
                    f"{key}={row[key]}"
                    for key in ("V", "E", "crossings")
                    if row.get(key) is not None
                )

                kg_text = _time_text(row, "knottedgraph")
                tp_text = _time_text(row, "topoly")
                if (
                    row.get("knottedgraph_status") == "ok"
                    and row.get("topoly_status") == "ok"
                    and row.get("knottedgraph_s")
                    and row.get("topoly_s")
                ):
                    speedup = float(row["topoly_s"]) / float(row["knottedgraph_s"])
                    speedup_text = f" | Topoly/KnottedGraph={speedup:.2f}x"
                else:
                    speedup_text = ""

                tqdm.write(
                    f"TIMING {row.get('family')} {meta} sample={sample_id} | "
                    f"KnottedGraph={kg_text} | Topoly={tp_text}"
                    f"{speedup_text}"
                )

                bar.set_postfix_str(
                    f"{row.get('family')} {meta} sample={sample_id} "
                    f"KG={kg_text}, T={tp_text}"
                )
                bar.update(1)
                continue
        tqdm.write(line)

    return_code = process.wait()
    bar.close()
    if return_code:
        raise RuntimeError(
            f"Benchmark failed with exit code {return_code}: {' '.join(map(str, cmd))}"
        )

    rows = summary_rows if summary_rows is not None else streamed_rows
    if not rows:
        raise RuntimeError("Benchmark completed without sample rows.")
    print(f"completed {len(rows)} sample records")
    return rows


## 2. Run the two paper benchmark families

### Crossing figure

For every $c$, the same graph-size panel is used. Graph sample $j$ is a connected trivalent graph built from a prism background with a controlled crossing motif spliced into it. The 21 paper samples have different $V$ and hence different $E$, with

$$
E=\frac{3V}{2}.
$$

Crucially, the list of $(V,E)$ pairs is identical at every $c$. This is a blocked design: graph size varies across samples but is not allowed to drift systematically with crossing count.

### Input-size figure

The second benchmark varies total $V$ using planar $K_4$ components with zero projected crossings.


In [ ]:
paper_script = ROOT / "dev" / "benchmark_topoly_paper_scaling.py"
env = dict(os.environ)
env["PYTHONPATH"] = os.pathsep.join([str(SRC), str(DEV)])
env["PYTHONNOUSERSITE"] = "1"

paper_module = _load_module(
    paper_script,
    "kg_topoly_paper_scaling_notebook_plan",
)
plan = paper_module.paper_plan(PROFILE, CROSSING_GRAPHS, K4_EMBEDDINGS)

assert set(plan) == {
    "crossings_graph_ensemble",
    "vertices_k4",
}

total = (
    len(plan["crossings_graph_ensemble"]["x_values"]) * CROSSING_GRAPHS
    + len(plan["vertices_k4"]["x_values"]) * K4_EMBEDDINGS
)

cmd = [
    sys.executable, str(paper_script),
    "--profile", PROFILE,
    "--crossing-graphs", str(CROSSING_GRAPHS),
    "--k4-embeddings", str(K4_EMBEDDINGS),
    "--timeout", str(TIMEOUT_S),
    "--seed", str(BASE_SEED),
    "--censor-frontier", str(CENSOR_FRONTIER),
]

rows = run_streamed_benchmark(
    cmd, total=total, description="Paper Yamada scaling"
)


## 3. Acceptance checks and raw-data export

The notebook verifies that the crossing benchmark really contains different graph sizes and that **the same $(V,E)$ panel occurs at every crossing count**. It also verifies exact trivalence, connectivity, the requested crossing count, and paired KnottedGraph/Topoly correctness whenever both frameworks finish.


In [ ]:
from collections import defaultdict

allowed = {"crossings_graph_ensemble", "vertices_k4"}
assert {row["family"] for row in rows} <= allowed

groups = defaultdict(list)
for row in rows:
    groups[(row["family"], int(row["size"]))].append(row)

expected_crossing_panel = None
for (family, size), group in sorted(groups.items()):
    if family == "crossings_graph_ensemble":
        assert len(group) == CROSSING_GRAPHS, (family, size, len(group))
        assert len({int(row["sample"]) for row in group}) == CROSSING_GRAPHS
        panel = sorted((int(row["V"]), int(row["E"])) for row in group)
        assert len(set(panel)) == CROSSING_GRAPHS
        assert all(int(row["crossings"]) == size for row in group)
        assert all(bool(row["connected"]) for row in group)
        assert all(int(row["regular_degree"]) == 3 for row in group)
        assert all(int(row["E"]) == 3 * int(row["V"]) // 2 for row in group)
        if expected_crossing_panel is None:
            expected_crossing_panel = panel
        else:
            assert panel == expected_crossing_panel, (size, panel, expected_crossing_panel)
    else:
        assert len(group) == K4_EMBEDDINGS, (family, size, len(group))
        assert len({int(row["embedding"]) for row in group}) == K4_EMBEDDINGS
        assert len({row["embedding_hash"] for row in group}) == K4_EMBEDDINGS
        assert all(int(row["V"]) == size for row in group)
        assert all(int(row["crossings"]) == 0 for row in group)

for row in rows:
    if row["correctness"] == "PASS":
        assert row["knottedgraph_status"] == "ok"
        assert row["topoly_status"] == "ok"
        assert row["pd_hash"]

keys = list(dict.fromkeys(key for row in rows for key in row))
with raw_csv.open("w", newline="") as handle:
    writer = csv.DictWriter(handle, fieldnames=keys)
    writer.writeheader()
    writer.writerows(rows)

print("PASS: only the two paper benchmark families were evaluated.")
print("PASS: crossing benchmark uses a fixed heterogeneous V/E panel across c.")
print("Crossing V/E panel:", expected_crossing_panel)
print(f"wrote {len(rows)} sample-level records to {raw_csv}")


## 4. Generate the two paper figures

Only two figure pairs are generated:

1. `topoly_vs_knottedgraph_crossings_fixed.{png,pdf}` — now the heterogeneous connected-trivalent crossing benchmark;
2. `topoly_vs_knottedgraph_vertices_k4.{png,pdf}` — the controlled input-size benchmark.


In [ ]:
plot_script = ROOT / "dev" / "plot_topoly_paper_scaling.py"
plot_cmd = [
    sys.executable, str(plot_script), str(raw_csv),
    "--figure-dir", str(FIG),
    "--aggregate-csv", str(aggregate_csv),
]

print("Running:", " ".join(plot_cmd))
plot_process = subprocess.Popen(
    plot_cmd, cwd=ROOT, env=env, text=True,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=1,
)
if plot_process.stdout is None:
    raise RuntimeError("Plot subprocess did not expose stdout.")
for line in plot_process.stdout:
    print(line, end="")
if plot_process.wait():
    raise RuntimeError("Paper-quality plot generation failed.")

expected_stems = [
    "topoly_vs_knottedgraph_crossings_fixed",
    "topoly_vs_knottedgraph_vertices_k4",
]
for stem in expected_stems:
    assert (FIG / f"{stem}.png").exists(), stem
    assert (FIG / f"{stem}.pdf").exists(), stem
assert aggregate_csv.exists()
print("PASS: both paper figure PNG/PDF pairs and aggregate CSV were created.")


## 5. Interpretation

**Crossing figure.** Each point aggregates over a fixed panel of heterogeneous connected trivalent graph sizes. Because the same $(V,E)$ panel is reused at every $c$, changes along the x-axis primarily track crossing complexity while averaging over graph-size heterogeneity. This is stronger than benchmarking only one $V=2,E=3$ theta graph, but it should still be described as a blocked heterogeneous-graph benchmark rather than a mathematically pure one-variable experiment.

**Input-size figure.** The $K_4$-component family varies total $V$ at zero projected crossings, giving a controlled graph/input-size benchmark.

The timed region contains only the Yamada evaluator after the common PD input has been prepared for both frameworks. Timeouts are censored rather than inserted into confidence intervals.
